# 60. 새 압력 전환 분기의 무작위성 확인·고정2-of-3 투표
59번 새 압력 전문가 RF/ExtraTrees의3개seed묶음을바꾼다. 압력참조30개·정상99%경계·방향조건은고정. 최고seed만선택하지않고3묶음중2묶음동의로새분기를결합한다. 온도앵커는54최고/57다수결을각각고정하므로전체모델모든요소의무작위검증과같지않다.
테스트를보고만든방향가설을같은자료로검사한다. 독립적성능향상/실제원인증명아니다. 과거→이후의악화와내부증거를별도로기록한다.


In [ ]:
from pathlib import Path
import json,pandas as pd,numpy as np
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
s59=''.join(json.loads((root/'notebooks/59_rg3_pressure_direction_confirmation.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
s54=''.join(json.loads((root/'notebooks/54_rg3_pdf_thermal_confirm_gates.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
function_source=s54[s54.index('def predict('):s54.index("models=['two_tree_mean")]
configs=[('seed42_7_101',[42,7,101]),('seed11_23_37',[11,23,37]),('seed13_29_53',[13,29,53])]
frames=[];mets=[]
for tag,seeds in configs:
 if tag=='seed42_7_101':
  pp=pd.read_csv(tables/'rg3_pressure_direction_predictions.csv',parse_dates=['TimeStamp'])
  mm=pd.read_csv(tables/'rg3_pressure_direction_metrics.csv')
 else:
  reset=function_source.replace('for seed in [42,7,101]:',f'for seed in {seeds}:')
  injected="exec(source.rsplit('records=[]',1)[0])\nexec("+repr(reset)+")\nmemo={}"
  modified=s59.replace("exec(source.rsplit('records=[]',1)[0])",injected).replace('rg3_pressure_direction_',f'rg3_pressure_seed_{tag}_')
  ns={'__name__':'__main__'};exec(modified,ns)
  pp=pd.read_csv(tables/f'rg3_pressure_seed_{tag}_predictions.csv',parse_dates=['TimeStamp'])
  mm=pd.read_csv(tables/f'rg3_pressure_seed_{tag}_metrics.csv')
 pp['configuration']=tag;mm['configuration']=tag
 frames.append(pp);mets.append(mm)
 print('Completed new pressure seed test',tag,flush=True)
p=pd.concat(frames,ignore_index=True);m=pd.concat(mets,ignore_index=True)
p.to_csv(tables/'rg3_pressure_seed_predictions.csv',index=False,encoding='utf-8-sig')
m.to_csv(tables/'rg3_pressure_seed_metrics.csv',index=False,encoding='utf-8-sig')
p=p[p.model.eq('pressure_fall_speed_rise_switch_steeper')]
rows=[];votes=[]
for (scheme,anchor_name),g in p.groupby(['scheme','anchor']):
 wide=g.pivot(index='TimeStamp',columns='configuration',values='prediction')
 assert wide.notna().all().all() and wide.shape[1]==3
 actual=g.groupby('TimeStamp').actual.first().loc[wide.index].to_numpy(int)
 anc=g.groupby('TimeStamp').anchor_prediction.first().loc[wide.index].to_numpy(bool)
 pr=anc|(wide.sum(axis=1).to_numpy()>=2)
 tn,fp,fn,tp=confusion_matrix(actual,pr,labels=[0,1]).ravel()
 rows.append(dict(scheme=scheme,anchor=anchor_name,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn)))
 for j,t in enumerate(wide.index):
  r=dict(scheme=scheme,anchor=anchor_name,TimeStamp=t,actual=int(actual[j]),prediction=int(pr[j]),anchor_prediction=int(anc[j]),positive_votes=int(wide.iloc[j].sum()))
  r.update({k:int(wide.iloc[j][k]) for k in wide.columns});votes.append(r)
pd.DataFrame(rows).to_csv(tables/'rg3_pressure_seed_majority_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(votes).to_csv(tables/'rg3_pressure_seed_majority_predictions.csv',index=False,encoding='utf-8-sig')
print('Fixed2-of-3 pressure majority; thermal anchor unchanged')
print(pd.DataFrame(rows).to_string(index=False))
print('Original pressure directional configuration and alternatives')
print(m[m.model.eq('pressure_fall_speed_rise_switch_steeper')].to_string(index=False))
